# FNO Darcy Fourier-mode study on a Kaggle GPU

Launcher for [physicsnemo-fno-darcy](https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

With `RUN_FULL = False` (the default) the notebook runs the tests, a CUDA smoke study and a one-pseudo-epoch timing probe, and stops. Set `RUN_FULL = True` to run the study of `configs/full.yaml`.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # only the value True starts the full study

import re
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-fno-darcy"

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git fetch --all --quiet && git checkout --quiet {REF}", cwd=REPO)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert not dirty, "the working tree is not clean"
print("commit", head)

The requirements pin PhysicsNeMo to a commit of its main branch. `pip` may replace the PyTorch build of the image if it is older than PhysicsNeMo requires; the next cell prints the versions afterwards, in a new process, and checks CUDA again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

In [ ]:
sh("python -m pytest -q", cwd=REPO)

## CUDA smoke study

Two tiny models, two pseudo-epochs. The numbers are not results.

In [ ]:
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Timing probe

Generates the datasets of the full study (they are cached and reused below) and trains the largest model for one pseudo-epoch. The projection is an extrapolation from that one pseudo-epoch.

In [ ]:
import json
sh("python scripts/train.py --config full name=full_probe study.fno_modes=[16] study.seeds=[0] "
   "training.max_pseudo_epochs=1 training.rec_results_freq=1 validation.validation_pseudo_epochs=1", cwd=REPO)
probe = json.load(open(f"{REPO}/runs/full_probe/modes16_seed0/train_metrics.json"))
meta = json.load(open(f"{REPO}/runs/full_probe/study_metadata.json"))
cfg = meta["config"]
n_runs = 12  # 4 mode counts x 3 seeds in configs/full.yaml
projected_h = probe["train_seconds"] * 60 * n_runs / 3600  # 60 pseudo-epochs in configs/full.yaml
print(f"one pseudo-epoch, 16 modes: {probe['train_seconds']:.1f} s on {meta['gpu_name']}")
print(f"data generation: " + ", ".join(f"{k} {v['generation_seconds']:.0f} s" for k, v in meta["datasets"].items()))
print(f"PROJECTION (upper estimate, every run costed as the 16-mode model): {projected_h:.1f} h for the full study")

In [ ]:
if RUN_FULL is True:
    sh("python scripts/train.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-fno-darcy-full.zip && cp -r runs/full/figures {WORK}/full_figures", cwd=REPO)
    print("FULL STUDY COMPLETE. Download /kaggle/working/physicsnemo-fno-darcy-full.zip")
else:
    print("KAGGLE CUDA SMOKE COMPLETE. No full study was run (RUN_FULL is False).")